In [3]:
import pandas as pd
import duckdb as db

In [4]:
df = pd.read_csv("../Data/clean_data.csv")

`Revenue gengerated by male and female`

In [6]:
df.columns

Index(['Unnamed: 0', 'customer_id', 'age', 'gender', 'item_purchased',
       'category', 'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'previous_purchases', 'payment_method',
       'frequency_of_purchases', 'age_group', 'purchases_frequency_days'],
      dtype='str')

In [10]:
db.sql("""

select gender, SUM(purchase_amount) as revenue 
       from df 
       group by gender

""")

┌─────────┬─────────┐
│ gender  │ revenue │
│ varchar │ int128  │
├─────────┼─────────┤
│ Male    │  157890 │
│ Female  │   75191 │
└─────────┴─────────┘

In [13]:
df['discount_applied'].value_counts()

discount_applied
No     2223
Yes    1677
Name: count, dtype: int64

2. which customer used a discount but still spent more then avg amount

In [15]:
result = db.sql("""
    SELECT customer_id, purchase_amount
    FROM df 
    WHERE discount_applied = 'Yes' 
      AND purchase_amount >= (SELECT AVG(purchase_amount) FROM df)
""").df()

result

,customer_id,purchase_amount
0,2,64
1,3,73
2,4,90
3,7,85
4,9,97
...,...,...
834,1667,64
835,1671,73
836,1673,73
837,1674,62


In [17]:
df['review_rating'].value_counts().head()

review_rating
3.4    182
4.0    181
4.6    170
4.2    169
3.7    168
Name: count, dtype: int64

which are top 5 product with heighest avg reiew rating 

In [22]:
df.columns

Index(['Unnamed: 0', 'customer_id', 'age', 'gender', 'item_purchased',
       'category', 'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'previous_purchases', 'payment_method',
       'frequency_of_purchases', 'age_group', 'purchases_frequency_days'],
      dtype='str')

In [28]:
db.sql("""

select item_purchased ,
        ROUND(AVG(review_rating),2) as avg_product_rating
from df 
group by item_purchased
order by AVG(review_rating) desc
limit 5       

""")

┌────────────────┬────────────────────┐
│ item_purchased │ avg_product_rating │
│    varchar     │       double       │
├────────────────┼────────────────────┤
│ Gloves         │               3.86 │
│ Sandals        │               3.84 │
│ Boots          │               3.82 │
│ Hat            │                3.8 │
│ Skirt          │               3.78 │
└────────────────┴────────────────────┘

3. COMPARE THE AVG PURCHASE AMOUNT BETWEEN STANDARD AND EXPRESS SHIPING 

In [29]:
df.columns

Index(['Unnamed: 0', 'customer_id', 'age', 'gender', 'item_purchased',
       'category', 'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'previous_purchases', 'payment_method',
       'frequency_of_purchases', 'age_group', 'purchases_frequency_days'],
      dtype='str')

In [36]:
db.sql("""
select shipping_type ,
    ROUND(AVG(purchase_amount), 2)
    from df 
    where shipping_type in ('Standard', 'Express')
    group by shipping_type

""").df()

,shipping_type,"round(avg(purchase_amount), 2)"
0,Standard,58.46
1,Express,60.48


5. DO SUBSCRIBE CUSTOMER SPEND MORE ? COMPARE AVG SPEND AND TOTAL SPEND BETWEEN SUBSCRIBE CUSTOMER AND NON_SUBSCIBE CUSTOMER

In [37]:
df.columns

Index(['Unnamed: 0', 'customer_id', 'age', 'gender', 'item_purchased',
       'category', 'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'previous_purchases', 'payment_method',
       'frequency_of_purchases', 'age_group', 'purchases_frequency_days'],
      dtype='str')

In [39]:
db.sql("""

select subscription_status, 
       COUNT(customer_id) AS total_customer,
       ROUND(AVG(purchase_amount), 2) as avg_spend,
       ROUND(SUM(purchase_amount), 2) as total_ravenue

       from df 

       group by subscription_status
       order by total_ravenue, avg_spend desc
       

""")

┌─────────────────────┬────────────────┬───────────┬───────────────┐
│ subscription_status │ total_customer │ avg_spend │ total_ravenue │
│       varchar       │     int64      │  double   │    int128     │
├─────────────────────┼────────────────┼───────────┼───────────────┤
│ Yes                 │           1053 │     59.49 │         62645 │
│ No                  │           2847 │     59.87 │        170436 │
└─────────────────────┴────────────────┴───────────┴───────────────┘

6. WHICH 5 PRODUCTS HAVE HEIGEST PERCENTAGE OF PURCHASE WITH DISCOUNT APPLIED

In [41]:
df.columns

Index(['Unnamed: 0', 'customer_id', 'age', 'gender', 'item_purchased',
       'category', 'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'previous_purchases', 'payment_method',
       'frequency_of_purchases', 'age_group', 'purchases_frequency_days'],
      dtype='str')

In [46]:
db.sql("""
SELECT item_purchased, 
       ROUND(SUM(CASE WHEN discount_applied = 'Yes' THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) AS discount_rate 
FROM df 
GROUP BY item_purchased 
ORDER BY discount_rate DESC 
LIMIT 5
""")


┌────────────────┬───────────────┐
│ item_purchased │ discount_rate │
│    varchar     │    double     │
├────────────────┼───────────────┤
│ Hat            │          50.0 │
│ Sneakers       │         49.66 │
│ Coat           │         49.07 │
│ Sweater        │         48.17 │
│ Pants          │         47.37 │
└────────────────┴───────────────┘

7. segment customer into new returning  and loyel based on thier total number of 
 previous purchase and show the count of each segment 

In [54]:
db.sql("""
    with customer_type as (
    select customer_id , previous_purchases,
    CASE
        WHEN previous_purchases = 1 THEN 'new'
        WHEN previous_purchases between 2 and 10 THEN 'returning'
        ELSE 'loyel'
        END AS customer_segment             
    from df
    )             
    select customer_segment , count(*) as number_of_customer
                       from customer_type 
    group by customer_segment
    order by number_of_customer desc
""")


┌──────────────────┬────────────────────┐
│ customer_segment │ number_of_customer │
│     varchar      │       int64        │
├──────────────────┼────────────────────┤
│ loyel            │               3116 │
│ returning        │                701 │
│ new              │                 83 │
└──────────────────┴────────────────────┘

8. what are the top 3 most purchased products within each categories

In [60]:
db.sql("""
WITH item_count AS (
    SELECT category, 
           item_purchased, 
           COUNT(customer_id) AS total_order,
           ROW_NUMBER() OVER (PARTITION BY category ORDER BY COUNT(customer_id) DESC) AS item_rank
    FROM df 
    GROUP BY category, item_purchased
)
SELECT category, item_purchased, total_order, item_rank
FROM item_count 
WHERE item_rank <= 3
ORDER BY category, item_rank
""").df()


,category,item_purchased,total_order,item_rank
0,Accessories,Jewelry,171,1
1,Accessories,Sunglasses,161,2
2,Accessories,Belt,161,3
3,Clothing,Pants,171,1
4,Clothing,Blouse,171,2
5,Clothing,Shirt,169,3
6,Footwear,Sandals,160,1
7,Footwear,Shoes,150,2
8,Footwear,Sneakers,145,3
9,Outerwear,Jacket,163,1


9. ARE CUSTOMER WHO ARE BUYERS (MORE THEN 5 PREVIOUS PURCHASE ) ASLO LIKELY TO SUBSCRIBE

In [63]:
df.columns

Index(['Unnamed: 0', 'customer_id', 'age', 'gender', 'item_purchased',
       'category', 'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'previous_purchases', 'payment_method',
       'frequency_of_purchases', 'age_group', 'purchases_frequency_days'],
      dtype='str')

In [ ]:
db.sql("""
select subscription_status,
count(customer_id)
       from df 
       where previous_purchases > 5
       group by subscription_status
""")

┌─────────────────────┬────────────────────┐
│ subscription_status │ count(customer_id) │
│       varchar       │       int64        │
├─────────────────────┼────────────────────┤
│ Yes                 │                958 │
│ No                  │               2518 │
└─────────────────────┴────────────────────┘

10. WHAT IS REVENUE CONTIBUTIONS OF EACH AGE GROUP 

In [67]:
db.sql("""

select age_group , sum(purchase_amount) as total_revenue 
from df 
       group by age_group 
       order by total_revenue desc

""")

┌─────────────┬───────────────┐
│  age_group  │ total_revenue │
│   varchar   │    int128     │
├─────────────┼───────────────┤
│ Young Adult │         62143 │
│ Middle Aged │         59197 │
│ Adult       │         55978 │
│ Senior      │         55763 │
└─────────────┴───────────────┘